In [1]:
pip install -U transformers accelerate torch pandas nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.7 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2

In [2]:
import re
import json
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
#Config
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GEN_CONFIG = {
    "max_new_tokens": 700,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.08,
}

THEMES = [
    "kindness",
    "honesty",
    "sharing",
    "teamwork",
    "patience",
    "taking responsibility",
    "helping a friend",
    "being brave in a small everyday situation",
    "not giving up",
    "listening to others",
]

In [4]:
# 2-3 few-shot examples is enough for a baseline
FEW_SHOT_EXAMPLES = [
    {
        "theme": "sharing",
        "output": """Title: Mira and the Mango Basket

Characters: Mira, Rafi, Grandma

Story:
Mira loved carrying Grandma's basket of mangoes from the garden to the kitchen.
One sunny morning, Grandma filled the basket with golden mangoes and asked Mira to help.
Mira smiled proudly and lifted the basket with both hands.

On the way home, Mira met her friend Rafi near the gate.
Rafi had been helping his father sweep leaves, and he looked hot and tired.
He stared at the basket and said, "Those mangoes look sweet."

Mira remembered how much she loved mangoes.
She wanted to keep all of them for her family.
For a moment, she held the basket a little tighter and kept walking.

But then she noticed Rafi wiping his forehead.
She stopped and thought about how Grandma always shared what they had.
Mira picked the ripest mango and handed it to Rafi with a smile.

Rafi's face lit up.
"Thank you, Mira!" he said.
They sat under the tree and talked while Rafi ate the mango.
When Mira reached home, Grandma asked why one mango was missing.

Mira told her the truth.
Grandma smiled warmly and said, "A shared fruit becomes sweeter."
Mira felt happy inside.
That evening, the whole family ate mango slices together, and Mira knew she had done the right thing.

Moral: Sharing what you have can make everyone happier."""
    },
    {
        "theme": "patience",
        "output": """Title: Niko and the Little Seed

Characters: Niko, Mama, Pip the sparrow

Story:
Niko found a tiny seed near the garden fence and ran to show Mama.
"Can we plant it today?" he asked.
Mama nodded, and together they placed the seed in soft brown soil.

The next morning, Niko rushed outside.
He looked at the pot and frowned.
Nothing had grown.
He poked the soil gently and sighed, "Why is it taking so long?"

Pip the sparrow landed on the fence and chirped.
Niko came back again at lunch, and then again before dinner.
Still nothing happened.
Niko crossed his arms and said, "Maybe this seed does not work."

Mama knelt beside him and said, "Some good things need time."
She helped him water the soil carefully.
The next day, Niko waited again.
This time he did not poke the dirt.
He only watered it and placed the pot where the sun could shine.

A few days later, a tiny green sprout peeked out.
Niko clapped and laughed.
Each morning, he cared for the plant a little more.
Soon it grew strong and tall.

Niko smiled at Pip and said, "I am glad I did not give up."
Mama hugged him and said, "Patience helps good things grow."
Niko looked proudly at the little plant dancing in the breeze.

Moral: Good things often take time, so patience is important."""
    },
]

In [5]:
# Data structures
@dataclass
class StoryEval:
    theme: str
    mode: str
    title_present: bool
    characters_present: bool
    story_present: bool
    moral_present: bool
    word_count: int
    within_word_limit: bool
    sentence_count: int
    avg_sentence_length_words: float
    estimated_scene_count: int
    scene_range_ok: bool
    character_count: int
    character_count_ok: bool
    banned_content_found: bool
    final_score: float
    output_text: str

In [6]:
# Load model
def load_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    return tokenizer, model

In [7]:
# Prompt builders
SYSTEM_PROMPT = """You are a children's story writer.

Write safe, warm, easy-to-understand stories for children ages 6 to 8.

Requirements:
- Keep the full story under 500 words
- Use simple vocabulary and clear sentences
- Use at most 2 to 3 main characters
- Make the story easy to divide into 6 or 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, romantic, dark, or adult themes

Output exactly in this format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

In [8]:
def build_zero_shot_messages(theme: str) -> List[Dict[str, str]]:
    user_prompt = f"Theme: {theme}\nWrite one original story."
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def build_few_shot_messages(theme: str) -> List[Dict[str, str]]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEW_SHOT_EXAMPLES:
        messages.append(
            {"role": "user", "content": f"Theme: {ex['theme']}\nWrite one original story."}
        )
        messages.append({"role": "assistant", "content": ex["output"]})

    messages.append(
        {"role": "user", "content": f"Theme: {theme}\nWrite one original story."}
    )
    return messages

In [9]:
#Generation
def generate_story(tokenizer, model, messages: List[Dict[str, str]]) -> str:
    # Hugging Face recommends using chat templates for chat/instruct models like Mistral-Instruct.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **GEN_CONFIG,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text

In [10]:
# Parsing helpers
def extract_section(text: str, label: str) -> Optional[str]:
    pattern = rf"{label}:\s*(.*?)(?=\n[A-Z][A-Za-z ]*:\s|\Z)"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else None


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))


def split_sentences(text: str) -> List[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sentences if s.strip()]


def estimate_scene_count(story_text: str) -> int:
    # Very rough proxy:
    # count sentences with strong event/action markers, capped loosely
    sentences = split_sentences(story_text)
    action_markers = 0
    for s in sentences:
        if re.search(
            r"\b(found|saw|went|ran|said|asked|helped|tried|looked|thought|gave|learned|decided|returned|shared|waited|grew|smiled|clapped|opened|carried)\b",
            s,
            flags=re.IGNORECASE,
        ):
            action_markers += 1
    return max(1, min(10, action_markers))


def parse_characters(characters_text: Optional[str]) -> List[str]:
    if not characters_text:
        return []
    parts = re.split(r",| and ", characters_text)
    cleaned = [p.strip() for p in parts if p.strip()]
    return cleaned


BANNED_PATTERNS = [
    r"\bkill\b",
    r"\bdead\b",
    r"\bblood\b",
    r"\bweapon\b",
    r"\bromance\b",
    r"\bkiss\b",
    r"\bghost\b",
    r"\bmonster\b",
    r"\bhorror\b",
    r"\bviolent\b",
]


In [11]:
def has_banned_content(text: str) -> bool:
    lower = text.lower()
    return any(re.search(p, lower) for p in BANNED_PATTERNS)


def evaluate_output(theme: str, mode: str, output_text: str) -> StoryEval:
    title = extract_section(output_text, "Title")
    characters = extract_section(output_text, "Characters")
    story = extract_section(output_text, "Story")
    moral = extract_section(output_text, "Moral")

    story_body = story if story else output_text
    word_count = count_words(story_body)
    sentences = split_sentences(story_body)
    sentence_count = len(sentences)
    avg_sentence_len = (
        round(word_count / sentence_count, 2) if sentence_count > 0 else 0.0
    )

    character_list = parse_characters(characters)
    est_scenes = estimate_scene_count(story_body)

    title_present = title is not None
    characters_present = characters is not None
    story_present = story is not None
    moral_present = moral is not None

    within_word_limit = word_count <= 500
    scene_range_ok = 6 <= est_scenes <= 7
    character_count_ok = 1 <= len(character_list) <= 3
    banned_found = has_banned_content(output_text)

    # Simple weighted score for baseline comparisons
    score = 0.0
    score += 10 if title_present else 0
    score += 10 if characters_present else 0
    score += 15 if story_present else 0
    score += 20 if moral_present else 0
    score += 15 if within_word_limit else 0
    score += 10 if scene_range_ok else 0
    score += 10 if character_count_ok else 0
    score += 10 if not banned_found else 0

    # prefer shorter sentence lengths for age 6-8
    if 6 <= avg_sentence_len <= 16:
        score += 10
    elif avg_sentence_len <= 20:
        score += 5

    return StoryEval(
        theme=theme,
        mode=mode,
        title_present=title_present,
        characters_present=characters_present,
        story_present=story_present,
        moral_present=moral_present,
        word_count=word_count,
        within_word_limit=within_word_limit,
        sentence_count=sentence_count,
        avg_sentence_length_words=avg_sentence_len,
        estimated_scene_count=est_scenes,
        scene_range_ok=scene_range_ok,
        character_count=len(character_list),
        character_count_ok=character_count_ok,
        banned_content_found=banned_found,
        final_score=round(score, 2),
        output_text=output_text,
    )

In [12]:
#Experiment runner
def run_experiment(tokenizer, model, themes: List[str]) -> pd.DataFrame:
    rows = []

    for theme in themes:
        # Zero-shot
        zs_messages = build_zero_shot_messages(theme)
        zs_output = generate_story(tokenizer, model, zs_messages)
        zs_eval = evaluate_output(theme, "zero_shot", zs_output)
        rows.append(asdict(zs_eval))

        # Few-shot
        fs_messages = build_few_shot_messages(theme)
        fs_output = generate_story(tokenizer, model, fs_messages)
        fs_eval = evaluate_output(theme, "few_shot", fs_output)
        rows.append(asdict(fs_eval))

    return pd.DataFrame(rows)

In [13]:
# Manual review helper
def add_manual_columns(df: pd.DataFrame) -> pd.DataFrame:
    manual_cols = {
        "manual_readability_1to5": None,
        "manual_coherence_1to5": None,
        "manual_moral_clarity_1to5": None,
        "manual_child_safety_1to5": None,
        "manual_scene_friendliness_1to5": None,
        "manual_notes": None,
    }
    for col, val in manual_cols.items():
        df[col] = val
    return df


# Main
def main():
    print(f"Loading model: {MODEL_ID}")
    tokenizer, model = load_model(MODEL_ID)

    print("Running baseline experiments...")
    df = run_experiment(tokenizer, model, THEMES)

    print("\n=== Summary by mode ===")
    summary = df.groupby("mode")[[
        "final_score",
        "word_count",
        "avg_sentence_length_words",
        "estimated_scene_count",
        "character_count",
    ]].mean()
    print(summary)

    # Save raw outputs and eval results
    df.to_csv("mistral_story_baselines.csv", index=False)

    # Save manual review sheet
    review_df = add_manual_columns(df.copy())
    review_df.to_csv("mistral_story_manual_review.csv", index=False)

    print("\nSaved:")
    print("- mistral_story_baselines.csv")
    print("- mistral_story_manual_review.csv")

    # Print a couple of samples
    print("\n=== Sample outputs ===")
    for mode in ["zero_shot", "few_shot"]:
        sample = df[df["mode"] == mode].iloc[0]
        print(f"\n--- {mode.upper()} | Theme: {sample['theme']} ---")
        print(sample["output_text"][:2000])


if __name__ == "__main__":
    main()

Loading model: mistralai/Mistral-7B-Instruct-v0.2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Running baseline experiments...

=== Summary by mode ===
           final_score  word_count  avg_sentence_length_words  \
mode                                                            
few_shot         103.0       191.9                     10.534   
zero_shot         99.0       258.2                     15.218   

           estimated_scene_count  character_count  
mode                                               
few_shot                     6.7              2.9  
zero_shot                    5.2              2.5  

Saved:
- mistral_story_baselines.csv
- mistral_story_manual_review.csv

=== Sample outputs ===

--- ZERO_SHOT | Theme: kindness ---
Title: The Kind Butterfly and the Grumpy Squirrel

Characters:
1. Sammy, a kind and curious little girl
2. Freddie, a grumpy old squirrel

Story:

Once upon a time in the beautiful Forest of Whispers, there lived a little girl named Sammy. She loved exploring the forest, discovering new things, and making friends with all its creatures. On